In [ ]:
# --- repo root + config (walk parents; do not use ../..) ---
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# CADEC model inference (8 models)

Run all baselines + matched-pair generative models over CADEC variants (original + accepted perturbations, **m ≥ 3** only).

- **Causal inference** (`generate_concept`): copied from `RQ3_matched_pairs.ipynb` (bf16, T=0 greedy, `device_map="auto"`, `free_model()` between models).
- **FLAN-T5**: seq2seq greedy, same loading policy.
- **Encoders**: SapBERT+FAISS + per-encoder re-rank → `output_text` = winning surface form (model-specific; avoids identical-across-models bug).

**Output:** `outputs/rq3/intermediate/rq3_cadec_model_outputs.csv`


In [1]:
# === Setup (Part 2) — absolute project root (nbconvert-safe) ===
import json
import os
import pickle
from pathlib import Path

import torch
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

PROJECT_ROOT = PROJECT_ROOT
CONFIG_PATH = PROJECT_ROOT / "config" / "config.json"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CONFIG_PATH:  {CONFIG_PATH}")
assert CONFIG_PATH.is_file(), f"Missing config.json: {CONFIG_PATH}"
assert torch.cuda.is_available(), "CUDA required — CPU placement is not allowed for this notebook"

with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)


def _expand_tree(obj):
    """Expand ~ in all string paths; leave non-strings / null unchanged."""
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _resolve_cfg_path(p):
    if p is None:
        return None
    path = Path(p)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


def _model_src(key: str) -> str:
    """Return local path or HF hub id for a model key in CFG['models']."""
    if key not in CFG["models"]:
        raise KeyError(f"Unknown model key {key!r}. Choose from: {sorted(CFG['models'])}")
    return CFG["models"][key]


def assert_model_on_cuda(model, name: str):
    """Crash loudly if the model landed on CPU.

    - Single-device models: first parameter must be cuda.
    - device_map="auto" models: at least one parameter must be on cuda.
    """
    devices = {p.device for p in model.parameters()}
    cuda_devs = {d for d in devices if d.type == "cuda"}
    assert cuda_devs, (
        f"CPU PLACEMENT BUG: {name} has no parameters on CUDA. "
        f"Devices seen: {sorted(str(d) for d in devices)}. "
        f"Refuse to run silently on CPU."
    )
    # For non-sharded models, require the primary device to be cuda
    first = next(model.parameters()).device
    if len(devices) == 1:
        assert first.type == "cuda", (
            f"CPU PLACEMENT BUG: {name} first parameter on {first}, expected cuda"
        )
    gpu_name = torch.cuda.get_device_name(0)
    print(f"DEVICE OK [{name}]: param_devices={sorted(str(d) for d in devices)} | GPU={gpu_name}")


def load_generative(key: str):
    """Causal 7–8B — bf16 + device_map=auto (RQ3_matched_pairs)."""
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    assert_model_on_cuda(model, f"causal:{key}")
    return tokenizer, model


def load_seq2seq(key: str):
    """FLAN-T5 etc. — NO device_map; explicit .to('cuda')."""
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
    )
    model = model.to("cuda").eval()
    assert_model_on_cuda(model, f"seq2seq:{key}")
    return tokenizer, model


def load_encoder(key: str):
    """BERT-family encoders — NO device_map; explicit .to('cuda')."""
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModel.from_pretrained(src)
    model = model.to("cuda").eval()
    assert_model_on_cuda(model, f"encoder:{key}")
    return tokenizer, model


def free_model(model):
    del model
    torch.cuda.empty_cache()


_pool_path = _resolve_cfg_path(CFG["pool_full"])
with open(_pool_path, "rb") as _f:
    cui_pool = pickle.load(_f)

_n_cuis = cui_pool.get("n_cuis", len(cui_pool.get("cuis", {})))
print(f"Config: {CONFIG_PATH}")
print(f"Loaded CUI pool from: {_pool_path}")
print(f"Pool type: {cui_pool.get('pool_type', 'unknown')} | unique CUIs: {_n_cuis:,}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
if _n_cuis < 10_000:
    print("WARNING: CUI count looks like the MeSH subset (~1,201), not full UMLS.")
else:
    print("Confirmed: full_umls-scale CUI pool loaded.")


/home/s224858267/.conda/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs
CONFIG_PATH:  /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/config.json


Config: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/config.json
Loaded CUI pool from: /home/s224858267/data/umls/pools/cui_pool_full_umls.pkl
Pool type: full_umls | unique CUIs: 3,341,331
GPU: NVIDIA L40S
Confirmed: full_umls-scale CUI pool loaded.


In [2]:
# Loud fail if wrong pool
_n_cuis = int(cui_pool.get("n_cuis", len(cui_pool.get("cuis", {}))))
print(f"Setup pool_type={cui_pool.get('pool_type')} | unique CUIs={_n_cuis:,}")
assert _n_cuis > 3_000_000, (
    f"Wrong CUI pool loaded: n_cuis={_n_cuis:,} (expected full_umls > 3,000,000)."
)
print("ASSERT OK: full_umls-scale pool loaded.")


Setup pool_type=full_umls | unique CUIs=3,341,331
ASSERT OK: full_umls-scale pool loaded.


## 1) Build CADEC variant table (m ≥ 3)


In [3]:
# === CADEC variants (original + accepted perts, m >= 3) ====================
import sys
import time
import gc
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()


INTER_DIR = PROJECT_ROOT / "outputs" / "rq3" / "intermediate"
INTER_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = INTER_DIR / "cadec_model_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = INTER_DIR / "rq3_cadec_model_outputs.csv"

CADEC_INST = INTER_DIR / "rq3_cadec_instances.csv"
CADEC_PERT = INTER_DIR / "rq3_cadec_perturbations.csv"
for label, p in [("CADEC instances", CADEC_INST), ("CADEC perturbations", CADEC_PERT)]:
    print(f"Path [{label}]: {p}")
    assert p.is_file(), f"Missing {label}: {p}"

MIN_M_ACCEPTED = 3
MAX_NEW_TOKENS = 32  # RQ3_matched_pairs
UNASSIGNED = "UNASSIGNED"

# Model registry (config.json keys)
ENCODER_MODELS = [
    {"key": "bert-base", "model_name": "BERT-base", "kind": "encoder", "domain": "general"},
    {"key": "biobert", "model_name": "BioBERT", "kind": "encoder", "domain": "biomedical"},
    {"key": "pubmedbert", "model_name": "PubMedBERT", "kind": "encoder", "domain": "biomedical"},
]
SEQ2SEQ_MODELS = [
    {"key": "flan-t5-base", "model_name": "FLAN-T5-base", "kind": "seq2seq", "domain": "general"},
]
CAUSAL_MODELS = [
    {"key": "biomistral", "model_name": "BioMistral-7B", "kind": "causal", "domain": "biomedical"},
    {"key": "mistral", "model_name": "Mistral-7B-Instruct-v0.1", "kind": "causal", "domain": "general"},
    {"key": "openbiollm", "model_name": "Llama3-OpenBioLLM-8B", "kind": "causal", "domain": "biomedical"},
    {"key": "llama3", "model_name": "Meta-Llama-3-8B-Instruct", "kind": "causal", "domain": "general"},
]
ALL_MODELS = ENCODER_MODELS + SEQ2SEQ_MODELS + CAUSAL_MODELS
MATCHED_PAIRS = [
    ("BioBERT", "BERT-base"),
    ("BioMistral-7B", "Mistral-7B-Instruct-v0.1"),
    ("Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct"),
]

print("Models to run:")
for s in ALL_MODELS:
    print(f"  {s['kind']:8s} {s['model_name']:28s} key={s['key']} src={_model_src(s['key'])}")

df_inst = pd.read_csv(CADEC_INST)
df_pert = pd.read_csv(CADEC_PERT)
df_pert = df_pert[df_pert["accepted_final"] == True].copy()
_log(f"CADEC instances={len(df_inst):,} | accepted pert rows={len(df_pert):,}")

_m = df_pert.groupby("instance_id").size().rename("m_accepted")
_keep = set(_m[_m >= MIN_M_ACCEPTED].index.astype(str))
_n_excl = int((~_m.index.astype(str).isin(_keep)).sum()) if len(_m) else 0
# also instances with zero accepted
_all_ids = set(df_inst["instance_id"].astype(str))
_n_excl_inst = len(_all_ids - _keep)
_log(
    f"m>=3 inclusion: keep={len(_keep):,} instances | "
    f"excluded={_n_excl_inst:,} | m distribution:"
)
print(_m.value_counts().sort_index().to_string())
sys.stdout.flush()

df_inst = df_inst[df_inst["instance_id"].astype(str).isin(_keep)].copy()
df_pert = df_pert[df_pert["instance_id"].astype(str).isin(_keep)].copy()

rows = []
for _, r in df_inst.iterrows():
    iid = r["instance_id"]
    ctx = r["mention_context"] if pd.notna(r.get("mention_context")) else r.get("original_text")
    rows.append({
        "instance_id": iid,
        "input_variant_id": f"{iid}_orig",
        "input_type": "original",
        "input_text": ctx,
        "gold_mention": r.get("gold_mention"),
        "gold_cui": r.get("gold_cui"),
        "perturbation_type": "original",
    })
for _, r in df_pert.iterrows():
    iid = r["instance_id"]
    rows.append({
        "instance_id": iid,
        "input_variant_id": f"{iid}_{r.get('perturbation_type', 'pert')}_{len(rows)}",
        "input_type": "perturbation",
        "input_text": r["perturbation_text"],
        "gold_mention": r.get("gold_mention"),
        "gold_cui": r.get("gold_cui"),
        "perturbation_type": r.get("perturbation_type"),
    })

df_variants = pd.DataFrame(rows)
# Stable unique variant ids per instance
_vid_rows = []
for iid, g in df_variants.groupby("instance_id", sort=False):
    for j, (_, r) in enumerate(g.iterrows()):
        d = r.to_dict()
        if d["input_type"] == "original":
            d["input_variant_id"] = f"{iid}_orig"
        else:
            d["input_variant_id"] = f"{iid}_p{j:02d}"
        _vid_rows.append(d)
df_variants = pd.DataFrame(_vid_rows)
_log(
    f"Variant table: {len(df_variants):,} rows | "
    f"instances={df_variants['instance_id'].nunique():,} | "
    f"mean variants/inst={df_variants.groupby('instance_id').size().mean():.2f}"
)
print(df_variants.groupby("input_type").size().to_string())
sys.stdout.flush()
assert df_variants["instance_id"].nunique() > 0


Path [CADEC instances]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_instances.csv
Path [CADEC perturbations]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_perturbations.csv
Models to run:
  encoder  BERT-base                    key=bert-base src=bert-base-uncased
  encoder  BioBERT                      key=biobert src=dmis-lab/biobert-v1.1
  encoder  PubMedBERT                   key=pubmedbert src=microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
  seq2seq  FLAN-T5-base                 key=flan-t5-base src=google/flan-t5-base
  causal   BioMistral-7B                key=biomistral src=/home/s224858267/data/models/BioMistral-7B
  causal   Mistral-7B-Instruct-v0.1     key=mistral src=/home/s224858267/data/models/Mistral-7B-Instruct-v0.1
  causal   Llama3-OpenBioLLM-8B         key=openbiollm src=/home/s224858267/data/models/Llama3-OpenBioLLM-8B
  causal  

[2026-07-30 05:30:19 UTC] m>=3 inclusion: keep=5,669 instances | excluded=1,337 | m distribution:


m_accepted
1     118
2    1083
3     440
4    2459
5     619
6    1880
7     115
8     156


[2026-07-30 05:30:21 UTC] Variant table: 33,253 rows | instances=5,669 | mean variants/inst=5.87


input_type
original         5669
perturbation    27584


## 2) Inference helpers (RQ3 causal `generate_concept` + seq2seq)


In [4]:
# === Causal generative inference ============================================
# Instruct models: apply_chat_template + short max_new_tokens (concept span).
# FLAN-T5 seq2seq path below is unchanged.

import re

CAUSAL_MAX_NEW_TOKENS = 16  # short concept span, not prose
_CONCEPT_INSTR = (
    "Identify the primary medical concept in the following clinical text. "
    "Reply with only the concept name.\n\n"
    "Text: {text}"
)
_STRIP_MARKERS = (
    "[/INST]",
    "</s>",
    "<s>",
    "Answer:",
    "Concept:",
    "The primary medical concept is",
)


def raw_path_for(key: str) -> Path:
    return RAW_DIR / f"cadec_raw_{key}.csv"


def maybe_skip_existing(spec: dict) -> Path | None:
    """Resume: if cadec_raw_<key>.csv exists and is non-empty, skip inference."""
    out_path = raw_path_for(spec["key"])
    if out_path.exists() and out_path.stat().st_size > 0:
        n_rows = sum(1 for _ in open(out_path, encoding="utf-8", errors="replace")) - 1
        _log(
            f"SKIP {spec['model_name']} — exists {out_path.name} "
            f"({out_path.stat().st_size:,} bytes, ~{n_rows:,} rows)"
        )
        return out_path
    return None


# Llama-3 chat template (OpenBioLLM ships without one)
_LLAMA3_CHAT_TEMPLATE = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    "+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}"
    "{{ content }}{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)


def _ensure_chat_template(tokenizer) -> None:
    """Attach a chat template when missing (OpenBioLLM = Llama-3 family)."""
    if getattr(tokenizer, "chat_template", None):
        return
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    unk = getattr(tokenizer, "unk_token_id", None)
    if eot is not None and eot != unk:
        tokenizer.chat_template = _LLAMA3_CHAT_TEMPLATE
        return
    # Mistral-style fallback
    tokenizer.chat_template = (
        "{{ bos_token }}{% for message in messages %}"
        "{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}"
        "{% elif message['role'] == 'assistant' %}{{ message['content'] }}"
        "{% endif %}{% endfor %}"
    )


def _clean_concept_output(decoded: str) -> str:
    """Strip residual chat-template / scaffolding markers from decoded span."""
    text = decoded.strip()
    for marker in ("[/INST]", "</s>", "<s>"):
        text = text.replace(marker, " ")
    for marker in ("Answer:", "Concept:", "The primary medical concept is"):
        if marker in text:
            text = text.split(marker)[-1]
    m = re.search(
        r"(?is)the primary medical concepts?\b.*?\b(?:is|are)\b\s*:?\s*",
        text,
    )
    if m:
        text = text[m.end():]
    text = text.strip(" \"'`.")
    text = " ".join(text.split()).strip()
    return text[:200]


def generate_concept(text: str, tokenizer, model) -> str:
    _ensure_chat_template(tokenizer)
    user_content = _CONCEPT_INSTR.format(text=text)
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512, padding=True)
    try:
        first_dev = next(model.parameters()).device
        enc = {k: v.to(first_dev) for k, v in enc.items()}
    except StopIteration:
        enc = {k: v.to("cuda:0") for k, v in enc.items()}

    input_len = enc["input_ids"].shape[-1]
    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    unk = getattr(tokenizer, "unk_token_id", None)
    if eot is not None and eot != unk and eot not in eos_ids:
        eos_ids.append(eot)
    gen_kwargs = dict(
        max_new_tokens=CAUSAL_MAX_NEW_TOKENS,
        do_sample=False,  # greedy, T=0
        pad_token_id=tokenizer.pad_token_id,
    )
    if eos_ids:
        gen_kwargs["eos_token_id"] = eos_ids if len(eos_ids) > 1 else eos_ids[0]
    with torch.no_grad():
        out = model.generate(**enc, **gen_kwargs)
    # Decode only newly generated tokens (avoids prompt echo)
    new_tokens = out[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return _clean_concept_output(decoded)


def generate_concept_seq2seq(text: str, tokenizer, model) -> str:
    """FLAN-T5 greedy decode — same clinical-concept task, seq2seq prompt."""
    prompt = (
        "Identify the primary medical concept in the following clinical text. "
        "Reply with only the concept name.\n\n"
        f"Text: {text}"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512, padding=True)
    # Explicit cuda — FLAN is loaded with .to("cuda"), not device_map
    enc = {k: v.to("cuda") for k, v in enc.items()}
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    with torch.no_grad():
        out = model.generate(**enc, **gen_kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()[:200]


def generate_concept_seq2seq_batch(texts, tokenizer, model, batch_size=32):
    """Batched FLAN-T5 greedy decode on CUDA."""
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) if t is not None else "" for t in texts[i:i + batch_size]]
        prompts = [
            "Identify the primary medical concept in the following clinical text. "
            "Reply with only the concept name.\n\n"
            f"Text: {t}"
            for t in batch
        ]
        enc = tokenizer(
            prompts, return_tensors="pt", truncation=True, max_length=512, padding=True
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        outs.extend([d.strip()[:200] for d in decoded])
    return outs


def _rows_from_generations(spec, gens):
    rows = []
    for (_, row), gen in zip(df_variants.iterrows(), gens):
        rows.append({
            "instance_id": row["instance_id"],
            "model_name": spec["model_name"],
            "input_variant_id": row["input_variant_id"],
            "input_type": row["input_type"],
            "output_text": gen,
            "gold_cui": row["gold_cui"],
            "perturbation_type": row["perturbation_type"],
        })
    return rows


def run_one_causal(spec: dict) -> Path:
    key, model_name = spec["key"], spec["model_name"]
    skipped = maybe_skip_existing(spec)
    if skipped is not None:
        return skipped
    out_path = raw_path_for(key)

    src = _model_src(key)
    _log(f"LOAD causal {model_name} from {src} (bf16, device_map=auto, T=0 greedy)")
    t0 = time.perf_counter()
    tokenizer, model = load_generative(key)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    _ensure_chat_template(tokenizer)

    gens = []
    for i, row in tqdm(df_variants.iterrows(), total=len(df_variants), desc=model_name):
        text = str(row["input_text"]) if pd.notna(row["input_text"]) else ""
        try:
            gens.append(generate_concept(text, tokenizer, model))
        except Exception as e:
            _log(f"WARN generate failed {model_name} row={i}: {e}")
            gens.append("")
        if (len(gens) % 500) == 0:
            _log(f"  {model_name}: {len(gens)}/{len(df_variants)} elapsed={time.perf_counter()-t0:.0f}s")

    df_out = pd.DataFrame(_rows_from_generations(spec, gens))
    df_out.to_csv(out_path, index=False)
    _log(f"DONE {model_name}: rows={len(df_out)} mean_len={df_out['output_text'].str.len().mean():.1f} -> {out_path.name}")

    free_model(model)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


def run_one_seq2seq(spec: dict) -> Path:
    key, model_name = spec["key"], spec["model_name"]
    skipped = maybe_skip_existing(spec)
    if skipped is not None:
        return skipped
    out_path = raw_path_for(key)

    src = _model_src(key)
    _log(f"LOAD seq2seq {model_name} from {src} (bf16, explicit cuda, T=0 greedy)")
    t0 = time.perf_counter()
    tokenizer, model = load_seq2seq(key)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    texts = df_variants["input_text"].fillna("").astype(str).tolist()
    _log(f"  Batched seq2seq generate on cuda, n={len(texts):,} batch_size=32")
    gens = generate_concept_seq2seq_batch(texts, tokenizer, model, batch_size=32)

    df_out = pd.DataFrame(_rows_from_generations(spec, gens))
    df_out.to_csv(out_path, index=False)
    _log(f"DONE {model_name}: rows={len(df_out)} mean_len={df_out['output_text'].str.len().mean():.1f} -> {out_path.name}")

    free_model(model)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


## 3) Encoder inference (SapBERT retrieve + encoder re-rank)


In [5]:
# === Encoder inference → concept string (Part-2 SapBERT+FAISS + encoder re-rank) ===
# Encoders do not generate text; output_text = winning UMLS surface form (model-specific).
# IMPORTANT: encoders + SapBERT run on CUDA via explicit .to("cuda") — never device_map=auto.
import faiss

EMB_DIR = Path.home() / "data" / "umls" / "embeddings" / "sapbert_full_len3"
SAPBERT_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
CONFIDENCE_THRESHOLD = 0.70
MIN_FORM_LEN = 3
ENCODER_BATCH_SIZE = 128  # batched GPU forwards

# TOP_K is LOCKED at config.yaml umls.faiss_top_k (=1000); k_selection.json must agree.
# Hard-fail instead of silently degrading: the previous `.get(..., 50)` / `else: TOP_K = 50`
# fallbacks would drop concept-lane retrieval depth 1000 -> 50 with no error if the file or
# the key went missing, changing every downstream margin and entropy number.
_FAISS_TOP_K_LOCKED = int(CFG_YAML["umls"]["faiss_top_k"])
_ksel_path = _resolve_cfg_path(CFG.get("k_selection", "outputs/rq1/k_selection.json"))
if _ksel_path is None or not Path(_ksel_path).exists():
    raise FileNotFoundError(
        f"k_selection.json not found at {_ksel_path!r}; refusing to guess TOP_K. "
        "It is a fixed external input (see RUN_ORDER.md)."
    )
with open(_ksel_path, "r", encoding="utf-8") as _f:
    _ksel = json.load(_f)
if "k_selected" not in _ksel:
    raise KeyError(f"'k_selected' missing from {_ksel_path}; refusing to guess TOP_K.")
TOP_K = int(_ksel["k_selected"])
if TOP_K != _FAISS_TOP_K_LOCKED:
    raise ValueError(
        f"TOP_K mismatch: k_selection.json k_selected={TOP_K} != "
        f"config.yaml umls.faiss_top_k={_FAISS_TOP_K_LOCKED} (LOCKED)."
    )
FAISS_TOP_K = TOP_K
_log(f"Encoder linker TOP_K={TOP_K} | ENCODER_BATCH_SIZE={ENCODER_BATCH_SIZE}")

_emb_npy = EMB_DIR / "embeddings.npy"
_forms_json = EMB_DIR / "surface_forms.json"
_pairs_json = EMB_DIR / "cui_form_pairs.json"
_index_path = EMB_DIR / "faiss.index"
for p in (_emb_npy, _forms_json, _pairs_json, _index_path):
    assert p.exists(), f"Missing SapBERT cache: {p}"

_form_embeddings = np.load(_emb_npy)
with open(_forms_json, "r", encoding="utf-8") as _f:
    _unique_forms = json.load(_f)
with open(_pairs_json, "r", encoding="utf-8") as _f:
    _form_cui_pairs = [tuple(x) for x in json.load(_f)]
_faiss_index = faiss.read_index(str(_index_path))

_form_to_cuis = defaultdict(set)
for _c, _f in _form_cui_pairs:
    _form_to_cuis[_f].add(_c)
_exact_index = defaultdict(set)
for _f, _cuis in _form_to_cuis.items():
    _exact_index[_f.casefold()].update(_cuis)
_raw_cuis = cui_pool["cuis"]
_cui_st21pv = {c: bool(rec.get("st21pv", False)) for c, rec in _raw_cuis.items()}

assert torch.cuda.is_available(), "CUDA required for encoder inference"
_sap_tok = AutoTokenizer.from_pretrained(SAPBERT_ID)
_sap_mdl = AutoModel.from_pretrained(SAPBERT_ID)
_sap_mdl = _sap_mdl.to("cuda").eval()
_sap_mdl.half()
assert_model_on_cuda(_sap_mdl, "SapBERT-query-encoder")


def _mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def embed_texts_model(model, tokenizer, texts, batch_size=ENCODER_BATCH_SIZE, max_len=64):
    """Batched mean-pool embeddings on CUDA. Inputs always moved to cuda."""
    assert next(model.parameters()).device.type == "cuda", (
        "embed_texts_model: model not on CUDA — refusing CPU forward"
    )
    vecs = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="embed", leave=False):
            batch = [str(t) if pd.notna(t) else "" for t in texts[i:i + batch_size]]
            enc = tokenizer(
                batch, return_tensors="pt", truncation=True, max_length=max_len, padding=True
            )
            enc = {k: v.to("cuda", non_blocking=True) for k, v in enc.items()}
            out = model(**enc)
            pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled.float(), p=2, dim=1)
            vecs.append(pooled.cpu().numpy())
    return np.vstack(vecs) if vecs else np.zeros((0, 768), dtype=np.float32)


def assign_encoder_output(query_text, mention_text, form_scores):
    """Return (output_text=surface_form_or_CUI, cui)."""
    cand = []
    for form, sc in form_scores.items():
        for cui in _form_to_cuis.get(form, ()):
            cand.append((cui, form, float(sc)))
    if not cand:
        return UNASSIGNED, UNASSIGNED

    exact_cuis = set()
    for key in [mention_text, query_text]:
        if key and str(key).strip():
            exact_cuis |= set(_exact_index.get(str(key).strip().casefold(), ()))
    if exact_cuis:
        exact_cand = [c for c in cand if c[0] in exact_cuis]
        if exact_cand:
            cand = exact_cand
        else:
            cand = [(c, str(mention_text), 1.0) for c in exact_cuis] + cand

    st_filt = [c for c in cand if _cui_st21pv.get(c[0], False)]
    if st_filt:
        cand = st_filt

    cand.sort(key=lambda x: -x[2])
    best_cui, best_form, best_sc = cand[0]
    if best_sc < CONFIDENCE_THRESHOLD:
        return UNASSIGNED, UNASSIGNED
    out = best_form if best_form and str(best_form).strip() else best_cui
    return str(out)[:200], best_cui


def run_one_encoder(spec: dict) -> Path:
    key, model_name = spec["key"], spec["model_name"]
    skipped = maybe_skip_existing(spec)
    if skipped is not None:
        return skipped
    out_path = raw_path_for(key)

    src = _model_src(key)
    _log(f"LOAD encoder {model_name} from {src} (explicit .to(cuda), NO device_map)")
    t0 = time.perf_counter()
    tokenizer, model = load_encoder(key)  # asserts CUDA inside

    texts = df_variants["input_text"].fillna("").astype(str).tolist()
    mentions = df_variants["gold_mention"].fillna("").astype(str).tolist()

    _log(f"  SapBERT FAISS retrieve TOP_K={FAISS_TOP_K} (batched GPU embed) …")
    q_sap = embed_texts_model(_sap_mdl, _sap_tok, texts, batch_size=ENCODER_BATCH_SIZE)
    D, I = _faiss_index.search(q_sap.astype(np.float32), FAISS_TOP_K)

    _log(f"  Encoder embed queries (batch={ENCODER_BATCH_SIZE}) …")
    q_enc = embed_texts_model(model, tokenizer, texts, batch_size=ENCODER_BATCH_SIZE, max_len=128)

    cand_forms = set()
    shortlists = []
    for i in range(len(texts)):
        forms_i = []
        for j in I[i]:
            if j < 0:
                continue
            f = _unique_forms[int(j)]
            if len(f) >= MIN_FORM_LEN:
                forms_i.append(f)
                cand_forms.add(f)
        shortlists.append(forms_i)

    form_list = sorted(cand_forms)
    _log(f"  Encoder embed {len(form_list):,} unique candidate forms (batch={ENCODER_BATCH_SIZE}) …")
    if form_list:
        form_vecs = embed_texts_model(
            model, tokenizer, form_list, batch_size=ENCODER_BATCH_SIZE, max_len=64
        )
        form_vec_map = {f: form_vecs[i] for i, f in enumerate(form_list)}
    else:
        form_vec_map = {}

    # Vectorised scoring: q · f^T per shortlist (GPU-friendly numpy)
    outs = []
    for i in range(len(texts)):
        forms_i = [f for f in shortlists[i] if f in form_vec_map]
        if not forms_i:
            outs.append(UNASSIGNED)
            continue
        F = np.stack([form_vec_map[f] for f in forms_i], axis=0)  # [n, d]
        scores = F @ q_enc[i]  # cosine since both L2-normalised
        form_scores = {f: float(s) for f, s in zip(forms_i, scores)}
        out_text, _cui = assign_encoder_output(texts[i], mentions[i], form_scores)
        outs.append(out_text)

    df_out = pd.DataFrame(_rows_from_generations(spec, outs))
    df_out.to_csv(out_path, index=False)
    _log(
        f"DONE {model_name}: rows={len(df_out)} "
        f"UNASSIGNED={(df_out['output_text']==UNASSIGNED).mean():.1%} "
        f"elapsed={time.perf_counter()-t0:.0f}s -> {out_path.name}"
    )

    free_model(model)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


[2026-07-30 05:30:21 UTC] Encoder linker TOP_K=1000 | ENCODER_BATCH_SIZE=128


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  43%|████▎     | 86/199 [00:00<00:00, 829.62it/s]

Loading weights:  85%|████████▍ | 169/199 [00:00<00:00, 671.91it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 800.12it/s]

DEVICE OK [SapBERT-query-encoder]: param_devices=['cuda:0'] | GPU=NVIDIA L40S


## 4) Run all 8 models (one at a time → `free_model()`)


In [6]:
# === Run all 8 models sequentially ==========================================
for spec in ENCODER_MODELS:
    run_one_encoder(spec)
for spec in SEQ2SEQ_MODELS:
    run_one_seq2seq(spec)
for spec in CAUSAL_MODELS:
    run_one_causal(spec)
_log("All 8 models processed — existing cadec_raw_*.csv files were skipped.")


[2026-07-30 05:32:06 UTC] SKIP BERT-base — exists cadec_raw_bert-base.csv (4,900,451 bytes, ~33,253 rows)


[2026-07-30 05:32:06 UTC] SKIP BioBERT — exists cadec_raw_biobert.csv (5,056,379 bytes, ~33,253 rows)


[2026-07-30 05:32:06 UTC] SKIP PubMedBERT — exists cadec_raw_pubmedbert.csv (5,248,559 bytes, ~33,253 rows)


[2026-07-30 05:32:06 UTC] SKIP FLAN-T5-base — exists cadec_raw_flan-t5-base.csv (3,832,800 bytes, ~33,253 rows)


[2026-07-30 05:32:06 UTC] SKIP BioMistral-7B — exists cadec_raw_biomistral.csv (3,964,921 bytes, ~33,253 rows)


...

Llama3-OpenBioLLM-8B: 100%|██████████| 33253/33253 [3:16:37<00:00,  2.82it/s]

[2026-07-30 08:49:01 UTC] DONE Llama3-OpenBioLLM-8B: rows=33253 mean_len=14.1 -> cadec_raw_openbiollm.csv


[2026-07-30 08:49:10 UTC] Freed Llama3-OpenBioLLM-8B


[2026-07-30 08:49:10 UTC] SKIP Meta-Llama-3-8B-Instruct — exists cadec_raw_llama3.csv (4,435,147 bytes, ~33,253 rows)


[2026-07-30 08:49:10 UTC] All 8 models processed — existing cadec_raw_*.csv files were skipped.


## 5) Export + matched-pair identical-output asserts


In [7]:
# === Concatenate → BioASQ-style outputs CSV + matched-pair asserts ==========
TARGET_COLS = [
    "instance_id",
    "model_name",
    "input_variant_id",
    "input_type",
    "output_text",
    "gold_cui",
    "perturbation_type",
]

parts = []
for spec in ALL_MODELS:
    p = RAW_DIR / f"cadec_raw_{spec['key']}.csv"
    assert p.is_file() and p.stat().st_size > 0, f"Missing model output: {p}"
    df_p = pd.read_csv(p)
    missing = [c for c in TARGET_COLS if c not in df_p.columns]
    assert not missing, f"{p.name} missing cols {missing}"
    parts.append(df_p[TARGET_COLS])
    _log(f"Loaded {p.name}: {len(df_p):,} rows")

df_all = pd.concat(parts, ignore_index=True)
print(f"Writing: {OUT_CSV}")
df_all.to_csv(OUT_CSV, index=False)
_log(f"Wrote {len(df_all):,} rows → {OUT_CSV}")
print("Columns:", list(df_all.columns))
print(df_all.groupby("model_name").size().to_string())
sys.stdout.flush()

# Assert matched pairs are not identical (identical-across-models bug signature)
print("\n===== Matched-pair output divergence asserts =====")
for bio_name, gen_name in MATCHED_PAIRS:
    a = df_all[df_all["model_name"] == bio_name].sort_values(["instance_id", "input_variant_id"])
    b = df_all[df_all["model_name"] == gen_name].sort_values(["instance_id", "input_variant_id"])
    assert len(a) > 0 and len(b) > 0, f"Missing outputs for pair {bio_name} vs {gen_name}"
    # Align on keys
    merged = a.merge(
        b,
        on=["instance_id", "input_variant_id", "input_type"],
        suffixes=("_bio", "_gen"),
        how="inner",
    )
    assert len(merged) > 0, f"No overlapping variants for {bio_name} vs {gen_name}"
    len_bio = merged["output_text_bio"].fillna("").astype(str).str.len().mean()
    len_gen = merged["output_text_gen"].fillna("").astype(str).str.len().mean()
    identical_frac = (
        merged["output_text_bio"].fillna("").astype(str)
        == merged["output_text_gen"].fillna("").astype(str)
    ).mean()
    print(
        f"{bio_name} vs {gen_name}: mean_len={len_bio:.2f} vs {len_gen:.2f} | "
        f"identical_frac={identical_frac:.4f} n={len(merged):,}"
    )
    # Fail on the identical-across-models bug signature:
    # (a) 100% identical strings, or (b) equal mean length AND >=99% identical.
    assert identical_frac < 1.0, (
        f"IDENTICAL-OUTPUT BUG: {bio_name} vs {gen_name} produced 100% identical output_text"
    )
    assert not (np.isclose(len_bio, len_gen) and identical_frac >= 0.99), (
        f"IDENTICAL-OUTPUT BUG signature: mean lengths equal "
        f"({len_bio:.6f} vs {len_gen:.6f}) and identical_frac={identical_frac:.4f} "
        f"for {bio_name} vs {gen_name}"
    )
_log("ASSERT OK: matched pairs are not identical-across-models.")


[2026-07-30 08:49:10 UTC] Loaded cadec_raw_bert-base.csv: 33,253 rows


[2026-07-30 08:49:10 UTC] Loaded cadec_raw_biobert.csv: 33,253 rows


[2026-07-30 08:49:10 UTC] Loaded cadec_raw_pubmedbert.csv: 33,253 rows


[2026-07-30 08:49:10 UTC] Loaded cadec_raw_flan-t5-base.csv: 33,253 rows


[2026-07-30 08:49:10 UTC] Loaded cadec_raw_biomistral.csv: 33,253 rows


...

[2026-07-30 08:49:11 UTC] Wrote 266,024 rows → /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_model_outputs.csv


Columns: ['instance_id', 'model_name', 'input_variant_id', 'input_type', 'output_text', 'gold_cui', 'perturbation_type']
model_name
BERT-base                   33253
BioBERT                     33253
BioMistral-7B               33253
FLAN-T5-base                33253
Llama3-OpenBioLLM-8B        33253
Meta-Llama-3-8B-Instruct    33253
Mistral-7B-Instruct-v0.1    33253
PubMedBERT                  33253



===== Matched-pair output divergence asserts =====
BioBERT vs BERT-base: mean_len=49.41 vs 42.86 | identical_frac=0.3890 n=33,253


BioMistral-7B vs Mistral-7B-Instruct-v0.1: mean_len=10.97 vs 11.50 | identical_frac=0.3424 n=33,253


Llama3-OpenBioLLM-8B vs Meta-Llama-3-8B-Instruct: mean_len=14.15 vs 14.17 | identical_frac=0.0915 n=33,253
[2026-07-30 08:49:11 UTC] ASSERT OK: matched pairs are not identical-across-models.
